In [18]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
import ipywidgets as widgets
from IPython.display import display, HTML

In [19]:
# Load the dataset
df = pd.read_csv('styles.csv')

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (44446, 12)


,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName,Unnamed: 10,Unnamed: 11
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt,NaN,NaN
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans,NaN,NaN
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch,NaN,NaN
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011.0,Casual,Manchester United Men Solid Black Track Pants,NaN,NaN
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt,NaN,NaN


In [20]:
# Select relevant columns
features = ['id', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'productDisplayName']
df = df[features]

In [21]:
# Handle missing values
df.fillna('', inplace=True)

In [22]:
# Combine textual features into a single column
df['combined_features'] = (
    df['masterCategory'] + " " +
    df['subCategory'] + " " +
    df['articleType'] + " " +
    df['baseColour'] + " " +
    df['productDisplayName']
)

In [23]:
# Create TF-IDF vectorizer
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = tfidf.fit_transform(df['combined_features'])

In [24]:
# Fit NearestNeighbors model (brute force cosine distance)
nn_model = NearestNeighbors(n_neighbors=6, metric='cosine', algorithm='brute')
nn_model.fit(tfidf_matrix)

NearestNeighbors(algorithm='brute', metric='cosine', n_neighbors=6)

In [31]:
def get_recommendations(product_name):

    if product_name not in df['productDisplayName'].values:
        print("Product not found")
        return pd.DataFrame()

    idx = df[df['productDisplayName'] == product_name].index[0]

    distances, indices = nn_model.kneighbors(tfidf_matrix[idx], n_neighbors=6)

    recommended_indices = indices.flatten()[1:]

    recommendations = df.iloc[recommended_indices]

    return recommendations[['productDisplayName','articleType','baseColour']]


In [32]:
# Create dropdown widget for product selection
product_dropdown = widgets.Dropdown(
    options=df['productDisplayName'].tolist(),
    description='Select Product:',
    style={'description_width': 'initial'},
    layout={'width': '500px'}
)

In [33]:
# Create button widget
recommend_button = widgets.Button(
    description='Get Recommendations',
    button_style='success',
    tooltip='Click to get recommendations',
    layout={'width': '200px'}
)

In [34]:
# Output widget to display results
output = widgets.Output()

In [35]:
# Button click event
def on_button_clicked(b):
    
    with output:
        
        output.clear_output()

        product_name = product_dropdown.value
        
        recommendations = get_recommendations(product_name)

        display(HTML(f"<h3>Recommended Products for: {product_name}</h3>"))
        
        display(recommendations)


In [36]:
# Link button to event handler
recommend_button.on_click(on_button_clicked)

In [37]:
# Display widgets
display(product_dropdown)
display(recommend_button)
display(output)

Dropdown(description='Select Product:', layout=Layout(width='500px'), options=('Turtle Check Men Navy Blue Shi…

Button(button_style='success', description='Get Recommendations', layout=Layout(width='200px'), style=ButtonSt…

Output()